# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhruvaiyer2-netizen/FlyRankMLProject/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q duckdb huggingface_hub

from huggingface_hub import login
login()  # paste your HF token

import duckdb
import pandas as pd
from huggingface_hub import HfFileSystem
duckdb.register_filesystem(HfFileSystem())

duckdb.sql("""
    CREATE OR REPLACE TABLE march_facts AS
    SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    WHERE month = '2026-03'
""")

pages_df = duckdb.sql("""
    SELECT
      content_hash_id,
      client_hash_id,
      SUM(gsc_impressions) AS impressions_90d,
      SUM(gsc_clicks) AS clicks_90d,
      SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
      100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM march_facts
    GROUP BY content_hash_id, client_hash_id
""").df()

pages_df["position_tier"] = pd.cut(
    pages_df["avg_position"], bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)

print(f"Loaded and aggregated: {len(pages_df)} page-level rows")
MIN_IMPRESSIONS = 500
reliable = pages_df[pages_df["impressions_90d"] >= MIN_IMPRESSIONS].copy()

print("=== SIGNAL A: CTR vs Position Tier ===")
signal_a = reliable.groupby("position_tier", observed=True).agg(
    n=("ctr", "size"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median")
)
print(signal_a)
print("Verdict: MIXED — mean declines by tier as expected, but median is flat between")
print("tiers 1-3 and 4-10, so position alone doesn't cleanly separate CTR.\n")

print("=== SIGNAL B: Impression Volume vs CTR Reliability ===")
buckets = [(1, 9), (10, 99), (100, 499), (500, 10_000_000)]
for lo, hi in buckets:
    b = pages_df[(pages_df["impressions_90d"] >= lo) & (pages_df["impressions_90d"] <= hi)]
    print(f"Impressions {lo}-{hi}: n={len(b)}, mean_ctr={b['ctr'].mean():.3f}, std_ctr={b['ctr'].std():.3f}")
print("Verdict: CONFIRMED — low-volume pages show extreme, unstable CTR.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded and aggregated: 331437 page-level rows
=== SIGNAL A: CTR vs Position Tier ===
                   n  mean_ctr  median_ctr
position_tier                             
1-3             7940  0.354214    0.235942
4-10           32300  0.319041    0.214592
11-20          10534  0.268571    0.166667
21+            11150  0.139617    0.075268
Verdict: MIXED — mean declines by tier as expected, but median is flat between
tiers 1-3 and 4-10, so position alone doesn't cleanly separate CTR.

=== SIGNAL B: Impression Volume vs CTR Reliability ===
Impressions 1-9: n=33532, mean_ctr=1.176, std_ctr=8.432
Impressions 10-99: n=41765, mean_ctr=0.364, std_ctr=1.520
Impressions 100-499: n=39517, mean_ctr=0.229, std_ctr=0.504
Impressions 500-10000000: n=61924, mean_ctr=0.283, std_ctr=0.353
Verdict: CONFIRMED — low-volume pages show extreme, unstable CTR.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Among pages with enough impressions to trust their CTR (≥500 in the month), group by position tier (1-3, 4-10, 11-20, 21+). Find each tier's median CTR — that's "normal" for that rank. Score each page by how far below its own tier's median it sits. A page below its tier median gets flagged for review; everything else is just monitored.

Reason codes this rule can output: ctr_below_tier_median — attached only when ctr_gap_score > 0. No other code exists; rows that don't qualify carry no reason code, since nothing is being claimed about them.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import os

MIN_IMPRESSIONS = 500
reliable = pages_df[pages_df["impressions_90d"] >= MIN_IMPRESSIONS].copy()

tier_median = reliable.groupby("position_tier", observed=True)["ctr"].median()
reliable["tier_median_ctr"] = reliable["position_tier"].map(tier_median).astype(float)
reliable["ctr_gap_score"] = reliable["tier_median_ctr"] - reliable["ctr"]

reliable["reason_code"] = reliable["ctr_gap_score"].apply(
    lambda g: "ctr_below_tier_median" if g > 0 else None
)
reliable["action"] = reliable["ctr_gap_score"].apply(
    lambda g: "review_title_meta" if g > 0 else "monitor"
)

queue = reliable.sort_values("ctr_gap_score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
queue[[
    "content_hash_id", "position_tier", "avg_position", "impressions_90d",
    "ctr", "tier_median_ctr", "ctr_gap_score", "reason_code", "action"
]].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Total reliable pages: {len(reliable)}")
print(f"Flagged for review: {(reliable['action'] == 'review_title_meta').sum()}")
print(f"Saved: {os.path.abspath('work/outputs/baseline_action_score.csv')}")

Total reliable pages: 61924
Flagged for review: 30958
Saved: /content/work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Pattern across all 20: every row ties at ctr_gap_score = 0.236 in tier 1-3 — these are all pages with real impressions but zero recorded clicks, the maximum possible gap for their tier. The ranking among them is arbitrary (tied scores, broken only by original row order), which is itself a limitation to flag in Section 4.

content_e24acb4e20aab4e2 — review_title_meta, tier 1-3, 786 impressions, 0 clicks. Confidence: low (near the 500 floor). Wrong if: this is a brand-new page still accumulating its first clicks, not a stuck one.
content_93b882b207e1d751 — 574 impressions, 0 clicks. Confidence: low. Wrong if: volume this close to the floor could still be a fluke month.
content_46bf98a63346a13f — 1,018 impressions, 0 clicks. Confidence: high. Wrong if: the title/snippet is fine but the result is being outranked visually by a featured snippet or ad above it.
content_892d9925f3f3e8f1 — 812 impressions, 0 clicks. Confidence: low. Wrong if: seasonal/low-intent query volume inflated impressions without real searcher interest.
content_2af42daa393a487b — 1,091 impressions, 0 clicks. Confidence: high. Wrong if: the page ranks for a navigational query where users already know the destination and don't need to click.
content_c017585e0b7f8e2e — 981 impressions, 0 clicks. Confidence: low. Wrong if: this is right at the volume boundary and could be a one-month anomaly.
content_c7d99dc8546e8470 — 690 impressions, 0 clicks. Confidence: low. Wrong if: data collection glitch rather than a true zero (worth a raw-row spot check).
content_8db435804565076b — 2,477 impressions, 0 clicks. Confidence: high. Wrong if: position is volatile day-to-day and this average masks brief, unrepresentative top-3 spikes.
content_5ea22b1a5a3aad91 — 818 impressions, 0 clicks. Confidence: low. Wrong if: impressions are borderline and skew the zero-click read.
content_54645a85e019a1a4 — 1,902 impressions, 0 clicks. Confidence: high. Wrong if: the query itself has naturally low click intent (e.g., answer shown directly in the SERP).
content_d7c6539a96cb16aa — 1,092 impressions, 0 clicks. Confidence: high. Wrong if: a site-wide tracking gap caused clicks to be undercounted, not a true zero.
content_1d6dc95ac1a6c2c1 — 699 impressions, 0 clicks. Confidence: low. Wrong if: too close to the floor to be a stable signal.
content_7e1edc4d2ba7c213 — 1,063 impressions, 0 clicks. Confidence: high. Wrong if: title/meta are actually fine and the issue is ranking volatility, not the listing itself.
content_16ef36a5d6a9cd36 — 2,668 impressions, 0 clicks. Confidence: high. Wrong if: this is a recent page still too new to have accumulated any clicks yet.
content_3ee625d5d582c59b — 609 impressions, 0 clicks. Confidence: low. Wrong if: borderline volume makes the zero less meaningful.
content_e3b8c66d9d75768b — 1,442 impressions, 0 clicks. Confidence: high. Wrong if: the page is cannibalized by another of the client's own pages ranking just above it.
content_f0e7418e56b0fb6b — 1,314 impressions, 0 clicks. Confidence: high. Wrong if: position is unstable rather than steady, undermining the tier assignment.
content_84149e9606a5c1aa — 610 impressions, 0 clicks. Confidence: low. Wrong if: near-floor volume makes this a weak signal.
content_6dffb550eab6b945 — 2,234 impressions, 0 clicks. Confidence: high. Wrong if: searcher intent for this query doesn't require a click (informational snippet fully answers it).
content_d1cd66436360eb4d — 1,482 impressions, 0 clicks. Confidence: high. Wrong if: a tracking/attribution issue zeroed out real clicks that happened elsewhere in the funnel.

In [6]:
def confidence_note(row, tier_sizes):
    if row["impressions_90d"] < 1000:
        return "low — volume close to the reliability floor"
    elif tier_sizes[row["position_tier"]] < 1000:
        return "medium — thinner tier sample, median less stable"
    else:
        return "high — strong volume, well-sampled tier"

tier_sizes = reliable["position_tier"].value_counts()
top20 = queue.head(20).copy()
top20["confidence"] = top20.apply(lambda r: confidence_note(r, tier_sizes), axis=1)

top20[[
    "content_hash_id", "action", "reason_code", "confidence",
    "ctr_gap_score", "impressions_90d", "position_tier"
]]

,content_hash_id,action,reason_code,confidence,ctr_gap_score,impressions_90d,position_tier
277102,content_e24acb4e20aab4e2,review_title_meta,ctr_below_tier_median,low — volume close to the reliability floor,0.235942,786.0,1-3
277099,content_93b882b207e1d751,review_title_meta,ctr_below_tier_median,low — volume close to the reliability floor,0.235942,574.0,1-3
277098,content_46bf98a63346a13f,review_title_meta,ctr_below_tier_median,"high — strong volume, well-sampled tier",0.235942,1018.0,1-3
277090,content_892d9925f3f3e8f1,review_title_meta,ctr_below_tier_median,low — volume close to the reliability floor,0.235942,812.0,1-3
277086,content_2af42daa393a487b,review_title_meta,ctr_below_tier_median,"high — strong volume, well-sampled tier",0.235942,1091.0,1-3
277083,content_c017585e0b7f8e2e,review_title_meta,ctr_below_tier_median,low — volume close to the reliability floor,0.235942,981.0,1-3
277079,content_c7d99dc8546e8470,review_title_meta,ctr_below_tier_median,low — volume close to the reliability floor,0.235942,690.0,1-3
108820,content_8db435804565076b,review_title_meta,ctr_below_tier_median,"high — strong volume, well-sampled tier",0.235942,2477.0,1-3
257118,content_5ea22b1a5a3aad91,review_title_meta,ctr_below_tier_median,low — volume close to the reliability floor,0.235942,818.0,1-3
78459,content_54645a85e019a1a4,review_title_meta,ctr_below_tier_median,"high — strong volume, well-sampled tier",0.235942,1902.0,1-3


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

9 of the top 20 are low-confidence, all due to impression volume sitting near the 500 floor rather than thin tiers — every position tier in this dataset has thousands of pages, so tier-median stability isn't a concern here. No product-decision flags (health_score, priority_score, etc.) appear anywhere in the rule's inputs. The source data is confined to March 2026 — no future window was used to build or validate the score.

In [7]:
# Weak picks: lowest-confidence entries in the top 20
weak = top20[top20["confidence"].str.startswith("low")]
print(f"Weak picks in top 20: {len(weak)}")
print(weak[["content_hash_id", "ctr_gap_score", "impressions_90d"]])

# Leakage check 1: confirm no product-decision flags were used anywhere in the rule
excluded_product_flags = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win", "refresh_tier"]
used_columns = list(reliable.columns)
leaked = [c for c in excluded_product_flags if c in used_columns]
print(f"\nProduct flags leaked into the rule: {leaked if leaked else 'none'}")

# Leakage check 2: confirm the source window is March 2026 only — no future data
window_check = duckdb.sql("SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest FROM march_facts").df()
print(f"\nSource window: {window_check['earliest'][0]} to {window_check['latest'][0]} — single month, no future data")

Weak picks in top 20: 9
                 content_hash_id  ctr_gap_score  impressions_90d
277102  content_e24acb4e20aab4e2       0.235942            786.0
277099  content_93b882b207e1d751       0.235942            574.0
277090  content_892d9925f3f3e8f1       0.235942            812.0
277083  content_c017585e0b7f8e2e       0.235942            981.0
277079  content_c7d99dc8546e8470       0.235942            690.0
257118  content_5ea22b1a5a3aad91       0.235942            818.0
214450  content_1d6dc95ac1a6c2c1       0.235942            699.0
214436  content_3ee625d5d582c59b       0.235942            609.0
132684  content_84149e9606a5c1aa       0.235942            610.0

Product flags leaked into the rule: none

Source window: 2026-03-01 00:00:00 to 2026-03-31 00:00:00 — single month, no future data


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.